# 06. Assign one main topic to each item

A paper, trial, or post can match several topics at once, which double counts it and makes the topics dependent on each other. To model counts per topic we need each item to belong to exactly one.

This notebook does that in two steps. First it merges topics that are really the same thing, so breast_lump and breast_cancer_screening both become breast_cancer. Then it applies a hierarchy, so when several topics apply the most specific diagnosis wins. A study matching UTI, period cramps, and breast cancer is a breast cancer study.

**Run from `notebooks/Reddit_Data/`.**

**Input:** the `reddit_posts`, `pubmed`, and `clinical_trials` tables
**Output:** `data/processed/one_topic/`, plus three staging tables

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

conn = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

## Pull each source in wide form

Each query returns one row per item per topic. `wide_topics` collapses that to one row per item with its topics as an array.

In [ ]:
def wide_topics(long_df: pd.DataFrame, id_cols: list, topic_col: str = "topic") -> pd.DataFrame:
    """Collapse one row per item per topic into one row per item."""
    return (
        long_df.groupby(id_cols)[topic_col]
        .apply(lambda x: np.array(sorted(set(x.dropna()))))
        .reset_index()
        .rename(columns={topic_col: "topics"})
    )


def get_reddit_wide(conn) -> pd.DataFrame:
    query = """
        SELECT
            rp.created_utc,
            rp.created,
            rt.topic
        FROM reddit_posts rp
        LEFT JOIN reddit_topics rt
            ON rp.created_utc = rt.created_utc
    """
    long_df = pd.read_sql(query, conn)
    # Non-topic columns get merged back after grouping
    meta = long_df[["created_utc", "created"]].drop_duplicates()
    wide = wide_topics(long_df, id_cols=["created_utc"])
    return wide.merge(meta, on="created_utc", how="left")


def get_pubmed_wide(conn) -> pd.DataFrame:
    query = """
        SELECT
            p.title,
            p.doi,
            p.year,
            k.topic
        FROM pubmed p
        LEFT JOIN pubmed_keywords pk
            ON p.title = pk.title AND p.doi = pk.doi
        LEFT JOIN keywords k
            ON pk.keyword = k.keyword
    """
    long_df = pd.read_sql(query, conn)
    meta = long_df[["title", "doi", "year"]].drop_duplicates()
    wide = wide_topics(long_df, id_cols=["title", "doi"])
    return wide.merge(meta, on=["title", "doi"], how="left")


def get_clinical_trials_wide(conn) -> pd.DataFrame:
    query = """
        SELECT
            ct.nct_number,
            ct.title,
            ct.start_year,
            k.topic
        FROM clinical_trials ct
        LEFT JOIN clinical_trials_keywords ctk
            ON ct.nct_number = ctk.nct_number
        LEFT JOIN keywords k
            ON ctk.keyword = k.keyword
    """
    long_df = pd.read_sql(query, conn)
    meta = long_df[["nct_number", "title", "start_year"]].drop_duplicates()
    wide = wide_topics(long_df, id_cols=["nct_number"])
    return wide.merge(meta, on="nct_number", how="left")

In [ ]:
reddit_wide = get_reddit_wide(conn)
pubmed_wide = get_pubmed_wide(conn)
trials_wide = get_clinical_trials_wide(conn)

for name, d in [("reddit", reddit_wide), ("pubmed", pubmed_wide), ("trials", trials_wide)]:
    print(f"{name}: {len(d)} rows")

## Topic groups

Which topics collapse into which. Most map to themselves. The ones that do not are cases where the split was not meaningful for our analysis.

In [ ]:
TOPIC_TO_GROUP = {
    "bartholin_cyst": "bartholin_cyst",
    "ovarian_torsion": "ovarian_torsion",
    "ovarian_cyst": "ovarian_cyst",
    "endometriosis": "endometriosis",
    "adenomyosis": "adenomyosis",
    "fibroadenoma": "fibroadenoma",
    "breast_lump": "breast_cancer",
    "breast_cancer_screening": "breast_cancer",
    "interstitial_cystitis": "interstitial_cystitis",
    "tubal_ligation": "tubal_ligation",
    "abortion_policy": "abortion_policy",
    "abortion": "abortion",

    "uti": "uti",
    "bacterial_vaginosis": "bacterial_vaginosis",
    "yeast_infection": "yeast_infection",
    "sti_std": "sti_std",

    "pcos": "pcos",
    "thyroid": "thyroid",
    "menopause": "menopause",
    "vulvodynia": "vulvodynia",

    "iud": "iud",
    "implant_shot_ring_patch": "implant_shot_ring_patch",
    "oral_contraceptives": "oral_contraceptives",
    "contraceptive_side_effects": "contraceptive_side_effects",

    "pregnancy_test": "pregnancy_test",

    "boric_acid": "boric_acid",
    "probiotics": "probiotics",
    "vaginal_ph_microbiome": "vaginal_ph_microbiome",
    "tampon_safety": "feminine_products",
    "menstrual_cup": "feminine_products",
    "period_tracking_apps": "period_tracking_apps",
    "covid_vaccine": "covid_vaccine",
    "dysmenorrhea": "dysmenorrhea",
    "amenorrhea": "amenorrhea",

    "heavy_bleeding": "heavy_bleeding",
    "abnormal_bleeding": "abnormal_bleeding",
    "irregular_periods": "irregular_periods",
    "pms_pmdd": "pms_pmdd",
    "unprotected_sex": "unprotected_sex",

    "pelvic_pain": "pelvic_floor",
    "pelvic_floor": "pelvic_floor",

    "painful_sex": "painful_sex",
    "breast_pain": "breast_pain",
    "libido": "libido",
    "hirsutism": "hirsutism",
    "hair_loss": "hair_loss",
    "iron_anemia": "iron_anemia",
    "menstrual_cycle": "menstrual_cycle",
    "hormonal_acne": "hormonal_acne",
    "bloating": "bloating",
    "nausea": "nausea",
    "fatigue_sleep": "fatigue_sleep",
    "headache_migraine": "headache_migraine",
    "allergic_reaction": "allergic_reaction",
    "heart_palpitations": "heart_palpitations",
    "hemorrhoids": "hemorrhoids",
    "hot_flashes_night_sweats": "hot_flashes_night_sweats",
}

## Hierarchy

Order matters. Earlier wins. Specific diagnoses sit at the top, general symptoms at the bottom, because a study that mentions both nausea and endometriosis is an endometriosis study.

In [ ]:
GROUP_PRIORITY = [
    "bartholin_cyst",
    "ovarian_torsion",
    "ovarian_cyst",
    "endometriosis",
    "adenomyosis",
    "fibroadenoma",
    "breast_cancer",
    "interstitial_cystitis",
    "tubal_ligation",
    "abortion_policy",
    "abortion",
    "uti",
    "bacterial_vaginosis",
    "yeast_infection",
    "sti_std",
    "pcos",
    "thyroid",
    "menopause",
    "vulvodynia",
    "iud",
    "implant_shot_ring_patch",
    "oral_contraceptives",
    "contraceptive_side_effects",
    "pregnancy_test",
    "boric_acid",
    "probiotics",
    "vaginal_ph_microbiome",
    "feminine_products",
    "period_tracking_apps",
    "covid_vaccine",
    "dysmenorrhea",
    "amenorrhea",
    "heavy_bleeding",
    "abnormal_bleeding",
    "irregular_periods",
    "pms_pmdd",
    "unprotected_sex",
    "pelvic_floor",
    "painful_sex",
    "breast_pain",
    "libido",
    "hirsutism",
    "hair_loss",
    "iron_anemia",
    "menstrual_cycle",
    "hormonal_acne",
    "bloating",
    "nausea",
    "fatigue_sleep",
    "headache_migraine",
    "allergic_reaction",
    "heart_palpitations",
    "hemorrhoids",
    "hot_flashes_night_sweats",
]

## Selection function

In [ ]:
def assign_main_topic(
    df: pd.DataFrame,
    topic_to_group: dict,
    group_priority: list,
    matched_topics_col: str = "topics",
    output_col: str = "main_topic",
) -> pd.DataFrame:
    """Assign the highest-priority group present in each row's topic array."""
    # Lower number means higher priority
    group_rank = {group: i for i, group in enumerate(group_priority)}

    unmapped_topics = set()
    unranked_groups = set()

    def resolve_row(topics) -> str:
        if topics is None or (isinstance(topics, float) and pd.isna(topics)):
            return None
        if len(topics) == 0:
            return None

        groups_present = []
        for t in topics:
            t_clean = str(t).strip().lower()
            group = topic_to_group.get(t_clean)
            if group is None:
                unmapped_topics.add(t_clean)
                continue
            groups_present.append(group)

        if not groups_present:
            return None

        def sort_key(group):
            if group in group_rank:
                return (0, group_rank[group])
            unranked_groups.add(group)
            return (1, group)  # unranked groups sort last, alphabetically

        groups_present.sort(key=sort_key)
        return groups_present[0]

    df = df.copy()
    df[output_col] = df[matched_topics_col].apply(resolve_row)

    if unmapped_topics:
        print(f"Warning: {len(unmapped_topics)} topic(s) not in TOPIC_TO_GROUP: {sorted(unmapped_topics)}")
    if unranked_groups:
        print(f"Warning: {len(unranked_groups)} group(s) not in GROUP_PRIORITY: {sorted(unranked_groups)}")

    return df

## Apply to all three sources

In [ ]:
from pathlib import Path

OUT_DIR = Path("../../data/processed/one_topic")
OUT_DIR.mkdir(parents=True, exist_ok=True)

trials_result = assign_main_topic(trials_wide, TOPIC_TO_GROUP, GROUP_PRIORITY)
pubmed_result = assign_main_topic(pubmed_wide, TOPIC_TO_GROUP, GROUP_PRIORITY)
reddit_result = assign_main_topic(reddit_wide, TOPIC_TO_GROUP, GROUP_PRIORITY)

trials_result.to_parquet(OUT_DIR / "clinical_trials_one_topic.parquet", index=False)
pubmed_result.to_parquet(OUT_DIR / "pubmed_one_topic.parquet", index=False)
reddit_result.to_parquet(OUT_DIR / "reddit_one_topic.parquet", index=False)

In [ ]:
# Items that matched no topic at all
for name, d in [("trials", trials_result), ("pubmed", pubmed_result), ("reddit", reddit_result)]:
    n_null = d["main_topic"].isna().sum()
    print(f"{name}: {len(d)} items, {n_null} with no topic ({n_null / len(d):.1%})")

## Load into the database

In [ ]:
def upload_topic_results(
    df: pd.DataFrame,
    id_cols: list,
    engine,
    table_name: str,
    topic_col: str = "main_topic",
) -> None:
    """Upload ids plus resolved topic to a staging table, dropping unresolved rows."""
    upload_df = df[id_cols + [topic_col]].rename(columns={topic_col: "topic"})
    upload_df = upload_df.dropna(subset=["topic"])
    upload_df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Uploaded {len(upload_df)} rows to {table_name}")


upload_topic_results(trials_result, ["nct_number"], conn, "staging_clinical_trials_topics")
upload_topic_results(pubmed_result, ["title", "doi"], conn, "staging_pubmed_topics")
upload_topic_results(reddit_result, ["created_utc"], conn, "staging_reddit_topics")